# Campus Central da UFRN em 3D com OSM2World

Este notebook baixa os dados do OpenStreetMap do Campus Central da UFRN, converte tudo em um modelo 3D com o [OSM2World](https://osm2world.org) e mostra o resultado em um visualizador interativo dentro do próprio Colab.

Fluxo do notebook

1. Instala o Java 17 e o OSM2World
2. Descobre automaticamente o polígono do campus no OSM (com opção de retângulo manual)
3. Baixa os dados pela API Overpass
4. Avalia quantos prédios têm altura ou número de andares mapeados
5. Gera o arquivo `.glb` (e opcionalmente o `.obj`) em um nível de detalhe leve
6. Abre o modelo em um visualizador 3D e disponibiliza os arquivos para download

Basta rodar as células em ordem (Ambiente de execução → Executar tudo).

## 1. Parâmetros

In [ ]:
# Nome usado para buscar o polígono da universidade no OSM
NOME_BUSCA = "Universidade Federal do Rio Grande do Norte"

# Região de busca ao redor de Natal (lat_min, lon_min, lat_max, lon_max)
REGIAO_NATAL = (-5.92, -35.30, -5.70, -35.15)

# Faixa, em metros, ao redor do limite do campus de onde as ruas também são baixadas
# (inclui o anel viário e as ruas do entorno). Use 0 para ficar só com o interior.
MARGEM_VIAS_M = 300

# Se quiser fixar o recorte manualmente, preencha (lat_min, lon_min, lat_max, lon_max).
# Deixe como None para usar a detecção automática.
BBOX_MANUAL = None
# Exemplo aproximado, confira no mapa antes de usar:
# BBOX_MANUAL = (-5.8455, -35.2120, -5.8295, -35.1945)

# Prédios até esta distância (m) do limite do campus também entram, mesmo estando do lado de fora
MARGEM_PREDIOS_M = 250

# "poligono" recorta pelo limite do campus; "retangulo" baixa tudo o que está no retângulo envolvente
# (use se a conferência da seção 5 mostrar prédios do campus que ficaram de fora)
RECORTE = "poligono"

# Nomes procurados na conferência da seção 5 (prédios que você sabe que existem no campus)
PROCURAR = ["Metrópole Digital", "IMD", "Restaurante Universitário"]

# Nível de detalhe do OSM2World, de 0 (mais leve) a 4 (mais detalhado e MUITO mais pesado).
LOD = 1

# Densidade de árvores geradas nas áreas de mata (árvores por m²). O padrão do OSM2World é 0.01.
ARVORES_POR_M2 = 0.001

# Gerar também o .obj (arquivo grande, útil para simuladores e alguns editores 3D)
GERAR_OBJ = False

# Acima deste tamanho o visualizador do Colab não é aberto, porque tende a travar a aba
LIMITE_VISUALIZADOR_MB = 40
FORCAR_VISUALIZADOR = False

# Memória máxima para a JVM
JAVA_XMX = "8g"

SAIDA_NOME = "ufrn_campus"

## 2. Instalação do Java 17 e do OSM2World

In [ ]:
import os, re, glob, shutil, subprocess, zipfile, pathlib, urllib.request

BASE = pathlib.Path("/content/osm2world_ufrn") if os.path.isdir("/content") else pathlib.Path.cwd() / "osm2world_ufrn"
BASE.mkdir(parents=True, exist_ok=True)
O2W_DIR = BASE / "OSM2World"
SAIDA_DIR = BASE / "saida"
SAIDA_DIR.mkdir(exist_ok=True)

def versao_java(bin_java):
    try:
        out = subprocess.run([bin_java, "-version"], capture_output=True, text=True).stderr
        m = re.search(r'version "(\d+)', out)
        return int(m.group(1)) if m else 0
    except FileNotFoundError:
        return 0

def achar_java17():
    candidatos = [shutil.which("java") or "java"] + sorted(glob.glob("/usr/lib/jvm/*/bin/java"), reverse=True)
    for c in candidatos:
        if versao_java(c) >= 17:
            return c
    return None

JAVA = achar_java17()
if JAVA is None:
    print("Instalando OpenJDK 17...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null",
                   shell=True, check=True)
    JAVA = achar_java17()
assert JAVA, "Não foi possível encontrar Java 17 ou superior."
print("Java usado:", JAVA, "| versão", versao_java(JAVA))

URL_O2W = "https://www.osm2world.org/download/files/latest/OSM2World-latest-bin.zip"
zip_path = BASE / "OSM2World-latest-bin.zip"
if not zip_path.exists():
    print("Baixando OSM2World...")
    urllib.request.urlretrieve(URL_O2W, zip_path)
if not O2W_DIR.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(O2W_DIR)

jars = sorted(O2W_DIR.rglob("OSM2World*.jar"), key=lambda p: len(str(p)))
assert jars, "OSM2World.jar não encontrado dentro do zip."
O2W_JAR = jars[0]
O2W_HOME = O2W_JAR.parent   # pasta onde ficam standard.properties e as texturas
print("OSM2World:", O2W_JAR)
print("Arquivos de estilo encontrados:", [p.name for p in O2W_HOME.glob("*.properties")])

## 3. Recorte do campus

A célula abaixo consulta a API Overpass, procura os polígonos marcados como `amenity=university` com o nome da UFRN na região de Natal e escolhe o maior deles, que deve ser o Campus Central. Os dados são recortados por esse polígono, e não por um retângulo, para que os quarteirões vizinhos fiquem de fora. O mapa logo em seguida mostra o limite para conferência. Se ele não estiver certo, preencha `BBOX_MANUAL` na célula de parâmetros e rode de novo.

In [ ]:
import json, time, urllib.parse

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]

def overpass(query, timeout=180):
    ultimo_erro = None
    for url in OVERPASS_URLS:
        try:
            dados = urllib.parse.urlencode({"data": query}).encode()
            req = urllib.request.Request(url, data=dados, headers={"User-Agent": "ufrn-osm2world-notebook"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read()
        except Exception as e:
            ultimo_erro = e
            print(f"  falhou em {url}: {e}")
            time.sleep(2)
    raise RuntimeError(f"Todas as instâncias Overpass falharam: {ultimo_erro}")

def detectar_bbox():
    s, w, n, e = REGIAO_NATAL
    q = f"""
    [out:json][timeout:90];
    (
      way["amenity"="university"]({s},{w},{n},{e});
      relation["amenity"="university"]({s},{w},{n},{e});
    );
    out body geom;
    """
    elems = json.loads(overpass(q))["elements"]
    alvo = NOME_BUSCA.lower()
    cands = []
    for el in elems:
        tags = el.get("tags", {})
        nomes = " ".join(v for k, v in tags.items() if "name" in k).lower()
        if alvo in nomes or "ufrn" in nomes:
            b = limites(el)
            area = (b["maxlat"] - b["minlat"]) * (b["maxlon"] - b["minlon"])
            cands.append((area, el))
    if not cands:
        return None
    cands.sort(key=lambda t: t[0], reverse=True)
    print("Polígonos encontrados (maior primeiro):")
    for area, el in cands[:5]:
        print(f"  {el['type']}/{el['id']}: {el.get('tags', {}).get('name', '(sem nome)')}  área≈{area:.6f} grau²")
    return cands[0][1]

def limites(el):
    """Retângulo envolvente do elemento; usa o campo bounds ou calcula a partir da geometria."""
    if "bounds" in el:
        return el["bounds"]
    pts = [p for linha in contornos(el) for p in linha]
    return {"minlat": min(p[0] for p in pts), "maxlat": max(p[0] for p in pts),
            "minlon": min(p[1] for p in pts), "maxlon": max(p[1] for p in pts)}

def contornos(el):
    """Lista de linhas [(lat, lon), ...] do limite do elemento, para desenhar no mapa."""
    if el["type"] == "way":
        return [[(p["lat"], p["lon"]) for p in el.get("geometry", [])]]
    return [[(p["lat"], p["lon"]) for p in mb["geometry"]]
            for mb in el.get("members", []) if mb.get("type") == "way" and mb.get("geometry")]

CAMPUS = None      # elemento do OSM usado como limite (None quando o recorte é manual)
CONTORNOS = []
if BBOX_MANUAL:
    BBOX = tuple(BBOX_MANUAL)
    print("Usando recorte retangular manual.")
else:
    CAMPUS = detectar_bbox()
    if CAMPUS is None:
        raise RuntimeError("Nenhum polígono da UFRN encontrado. Preencha BBOX_MANUAL na célula de parâmetros.")
    b = limites(CAMPUS)
    BBOX = (b["minlat"], b["minlon"], b["maxlat"], b["maxlon"])
    CONTORNOS = contornos(CAMPUS)
    print(f"Recorte pelo polígono {CAMPUS['type']}/{CAMPUS['id']}.")

S, W, N, E = BBOX
print(f"\nRetângulo envolvente (lat_min, lon_min, lat_max, lon_max) = ({S:.5f}, {W:.5f}, {N:.5f}, {E:.5f})")
largura_km, altura_km = (E - W) * 111.3, (N - S) * 111.3
print(f"Tamanho aproximado: {largura_km:.2f} km × {altura_km:.2f} km")
if max(largura_km, altura_km) > 5:
    print("Atenção: o recorte ficou grande. Confira no mapa e, se for o caso, use BBOX_MANUAL.")

In [ ]:
import folium

def mapa_base():
    """Mapa com fundos da Esri, que não exigem chave de API nem bloqueiam o Colab."""
    esri = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
    mp = folium.Map(location=[(S + N) / 2, (W + E) / 2], zoom_start=15, tiles=None)
    folium.TileLayer(esri.format("World_Street_Map"), attr="Tiles © Esri", name="Ruas").add_to(mp)
    folium.TileLayer(esri.format("World_Imagery"), attr="Tiles © Esri", name="Satélite", show=False).add_to(mp)
    return mp

m = mapa_base()
if CONTORNOS:
    for linha in CONTORNOS:
        folium.PolyLine(linha, color="#d62728", weight=3, tooltip="Limite do campus").add_to(m)
else:
    folium.Rectangle([[S, W], [N, E]], color="#d62728", weight=3, fill=False,
                     tooltip="Área que será convertida").add_to(m)
folium.LayerControl().add_to(m)
m

## 4. Download dos dados do OpenStreetMap

In [ ]:
OSM_FILE = BASE / f"{SAIDA_NOME}.osm"

# Relações entram só quando são prédios (multipolígonos com pátio interno, por exemplo).
# Baixar todas as relações traria linhas de ônibus e limites municipais inteiros,
# com ruas da cidade toda, e era isso que deixava o modelo enorme.
RELACOES = """
  rel(bw.v)["type"="multipolygon"]["building"];
  rel(bw.v)["type"="building"];
"""

def consulta_poligono(margem_m):
    tipo = "rel" if CAMPUS["type"] == "relation" else "way"
    vias = f'way(around.b:{margem_m})["highway"];' if margem_m > 0 else ""
    if margem_m > 0 and MARGEM_PREDIOS_M > 0:
        vias += f' way(around.b:{MARGEM_PREDIOS_M})["building"];'
    # pontos com nome no entorno (restaurantes, lojas), usados para identificar os prédios no visualizador
    pontos = f'node(around.b:{MARGEM_PREDIOS_M})["name"];' if margem_m > 0 and MARGEM_PREDIOS_M > 0 else ""
    borda = "way(r.c)->.b;" if tipo == "rel" else ".c->.b;"   # linhas que formam o limite do campus
    return f"""
    [out:xml][timeout:300];
    {tipo}({CAMPUS['id']})->.c;
    {borda}
    .c map_to_area->.a;
    ( way(area.a); {vias} )->.v;
    ( node(area.a); {pontos} .v; {RELACOES} );
    (._;>;);
    out meta;
    """

def consulta_bbox():
    return f"""
    [out:xml][timeout:300];
    way({S},{W},{N},{E})->.v;
    ( node({S},{W},{N},{E}); .v; {RELACOES} );
    (._;>;);
    out meta;
    """

def tem_dados(xml_bytes):
    return xml_bytes.count(b"<way ") > 10

tentativas = []
if CAMPUS is not None and RECORTE == "poligono":
    tentativas.append(("polígono do campus + vias e prédios do entorno", consulta_poligono(MARGEM_VIAS_M)))
    tentativas.append(("polígono do campus", consulta_poligono(0)))
tentativas.append(("retângulo envolvente", consulta_bbox()))

dados = None
for nome, q in tentativas:
    print(f"Baixando dados do OSM pelo {nome} (pode levar um ou dois minutos)...")
    try:
        r = overpass(q, timeout=400)
    except RuntimeError as e:
        print("  não funcionou:", e)
        continue
    if tem_dados(r):
        dados = r
        print("  ok")
        break
    print("  a consulta voltou vazia, tentando a próxima forma de recorte")
assert dados, "Não foi possível baixar os dados do OSM. Tente de novo em alguns minutos."

OSM_FILE.write_bytes(dados)
tam = OSM_FILE.stat().st_size / 1e6
print(f"Arquivo salvo: {OSM_FILE}  ({tam:.1f} MB, {dados.count(b'<way ')} vias e polígonos)")

## 5. Qualidade do mapeamento dos prédios

O OSM2World usa tags como `height`, `building:levels`, `roof:shape` e `building:colour` para dar forma aos prédios. Quando elas faltam, o prédio vira uma caixa com altura padrão. A célula abaixo mostra quantos prédios do recorte têm essas informações e lista os maiores sem altura, que são bons candidatos para completar no editor iD do OpenStreetMap.

In [ ]:
import xml.etree.ElementTree as ET
import pandas as pd

raiz = ET.parse(OSM_FILE).getroot()
nos = {n.get("id"): (float(n.get("lat")), float(n.get("lon"))) for n in raiz.iter("node")}

def area_aprox_m2(refs):
    pts = [nos[r] for r in refs if r in nos]
    if len(pts) < 3:
        return 0.0
    import math
    lat0 = math.radians(sum(p[0] for p in pts) / len(pts))
    xy = [(p[1] * 111320 * math.cos(lat0), p[0] * 110540) for p in pts]
    return abs(sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(xy, xy[1:] + xy[:1]))) / 2

linhas = []
for w in raiz.iter("way"):
    tags = {t.get("k"): t.get("v") for t in w.iter("tag")}
    if "building" not in tags:
        continue
    refs = [nd.get("ref") for nd in w.iter("nd")]
    linhas.append({
        "osm_id": int(w.get("id")),
        "nome": tags.get("name", ""),
        "building": tags["building"],
        "height": tags.get("height"),
        "levels": tags.get("building:levels"),
        "roof_shape": tags.get("roof:shape"),
        "colour": tags.get("building:colour"),
        "area_m2": round(area_aprox_m2(refs)),
    })

df = pd.DataFrame(linhas)
total = len(df)
if total:
    com_altura = df["height"].notna() | df["levels"].notna()
    print(f"Prédios no recorte: {total}")
    print(f"  com height ou building:levels: {com_altura.sum()} ({100 * com_altura.mean():.0f}%)")
    print(f"  com roof:shape:               {df['roof_shape'].notna().sum()}")
    print(f"  com building:colour:          {df['colour'].notna().sum()}")
    sem = df[~com_altura].sort_values("area_m2", ascending=False).head(15).copy()
    sem["editar_no_osm"] = sem["osm_id"].map(lambda i: f"https://www.openstreetmap.org/edit?way={i}")
    display(sem[["osm_id", "nome", "building", "area_m2", "editar_no_osm"]])
else:
    print("Nenhum prédio encontrado no recorte.")

### Conferência do recorte

A célula abaixo compara o que foi baixado com todos os prédios mapeados no retângulo envolvente do campus. Ela responde duas perguntas. A primeira é quais prédios existem no OSM e ficaram fora do recorte. A segunda é se os nomes da lista `PROCURAR` existem no OSM e se estão marcados como prédio. Um lugar que aparece aqui sem a tag `building` está mapeado só como ponto ou área, e por isso não vira volume 3D. Um lugar que não aparece não está mapeado e precisa ser desenhado no editor do OpenStreetMap.

In [ ]:
ids_baixados = {("way", w.get("id")) for w in raiz.iter("way")} | {("relation", r.get("id")) for r in raiz.iter("relation")} \
             | {("node", n.get("id")) for n in raiz.iter("node")}
regex = "|".join(re.escape(t) for t in PROCURAR) or "^$"
q = f"""
[out:json][timeout:120];
nwr["building"]({S},{W},{N},{E});
out tags center;
nwr["name"~"{regex}",i]({S},{W},{N},{E});
out tags center;
"""
elems = json.loads(overpass(q))["elements"]

def linha(el):
    t = el.get("tags", {})
    c = el.get("center", el)
    return {"tipo": el["type"], "osm_id": el["id"], "nome": t.get("name", ""),
            "building": t.get("building", "(não é prédio)"),
            "no_recorte": (el["type"], str(el["id"])) in ids_baixados,
            "lat": c.get("lat"), "lon": c.get("lon"),
            "link": f"https://www.openstreetmap.org/{el['type']}/{el['id']}"}

todos = pd.DataFrame([linha(e) for e in elems]).drop_duplicates(["tipo", "osm_id"])
predios = todos[todos["building"] != "(não é prédio)"]
fora = predios[~predios["no_recorte"]]
print(f"Prédios mapeados no retângulo envolvente: {len(predios)}")
print(f"  dentro do recorte: {int(predios['no_recorte'].sum())}")
print(f"  fora do recorte:   {len(fora)} (inclui a vizinhança do campus; {int((fora['nome'] != '').sum())} têm nome)")
if (fora["nome"] != "").any():
    print("\nPrédios com nome que ficaram fora do recorte:")
    display(fora[fora["nome"] != ""][["tipo", "osm_id", "nome", "building", "link"]])

achados = todos[todos["nome"].str.contains(regex, case=False, regex=True)]
print(f"\nElementos do OSM com os nomes procurados {PROCURAR}:")
if len(achados):
    display(achados[["tipo", "osm_id", "nome", "building", "no_recorte", "link"]])
else:
    print("  nenhum. Esses lugares não têm nome no OSM ou não estão mapeados.")

mc = mapa_base()
for l in CONTORNOS:
    folium.PolyLine(l, color="#d62728", weight=2).add_to(mc)
for _, r in predios.iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=3, weight=0, fill=True, fill_opacity=0.8,
                        fill_color="#1f77b4" if r["no_recorte"] else "#ff7f0e",
                        tooltip=(r["nome"] or f"{r['tipo']}/{r['osm_id']}")).add_to(mc)
folium.LayerControl().add_to(mc)
print("\nMapa: azul = prédio incluído no modelo, laranja = prédio mapeado que ficou de fora.")
mc

## 6. Conversão para 3D com o OSM2World

In [ ]:
# limpa resultados de execuções anteriores
shutil.rmtree(SAIDA_DIR, ignore_errors=True)
SAIDA_DIR.mkdir(exist_ok=True)

GLB = (SAIDA_DIR / f"{SAIDA_NOME}.glb").resolve()
OBJ_DIR = (BASE / "saida_obj").resolve()
shutil.rmtree(OBJ_DIR, ignore_errors=True)
OBJ = OBJ_DIR / f"{SAIDA_NOME}.obj"

# Arquivo de configuração: herda o estilo padrão e reduz a densidade de árvores
CONFIG = O2W_HOME / "ufrn.properties"
linhas_cfg = []
if (O2W_HOME / "standard.properties").exists():
    linhas_cfg.append("include = standard.properties")
linhas_cfg.append(f"treesPerSquareMeter = {ARVORES_POR_M2}")
CONFIG.write_text("\n".join(linhas_cfg) + "\n", encoding="utf-8")

jvm = [JAVA, f"-Xmx{JAVA_XMX}", "-Djava.awt.headless=true",
       "--add-exports", "java.base/java.lang=ALL-UNNAMED",
       "--add-exports", "java.desktop/sun.awt=ALL-UNNAMED",
       "--add-exports", "java.desktop/sun.java2d=ALL-UNNAMED",
       "-jar", str(O2W_JAR)]

def rodar(args):
    print("$ osm2world", " ".join(args))
    # roda dentro da pasta do OSM2World para que o estilo e as texturas sejam encontrados
    r = subprocess.run(jvm + args, cwd=O2W_HOME, capture_output=True, text=True)
    log = r.stdout + r.stderr
    (BASE / "osm2world.log").write_text(log, encoding="utf-8")
    n_erros = log.count("ERROR[")
    print(f"  código de saída {r.returncode}; {n_erros} objeto(s) do mapa pulados por erro de geometria "
          f"(log completo em {BASE / 'osm2world.log'})")
    if r.returncode != 0:
        print(log[-2000:])
    return r.returncode

saidas = [str(GLB)] + ([str(OBJ)] if GERAR_OBJ else [])
if GERAR_OBJ:
    OBJ_DIR.mkdir(exist_ok=True)

# Versões recentes usam o subcomando "convert"; a 0.4.0 usa só as opções.
codigo = rodar(["convert", "-i", str(OSM_FILE.resolve()), "-o", *saidas,
                "--lod", str(LOD), "--config", CONFIG.name])
if codigo != 0:
    print("Tentando a sintaxe antiga (0.4.x), sem controle de nível de detalhe...")
    saidas[0] = str(SAIDA_DIR / f"{SAIDA_NOME}.gltf")
    codigo = rodar(["-i", str(OSM_FILE.resolve()), "-o", *saidas, "--config", CONFIG.name])

for pasta in (SAIDA_DIR, OBJ_DIR):
    for p in sorted(pasta.rglob("*")) if pasta.exists() else []:
        if p.is_file() and p.stat().st_size > 1e5:
            print(f"  {p.name}  {p.stat().st_size / 1e6:.1f} MB")

modelo = GLB if GLB.exists() else next(iter(SAIDA_DIR.glob("*.gltf")), None)
assert modelo is not None, "Nenhum arquivo glb/gltf foi gerado. Veja o log da conversão."
MODELO_MB = modelo.stat().st_size / 1e6
if MODELO_MB > LIMITE_VISUALIZADOR_MB:
    print(f"\nAtenção: o modelo ficou com {MODELO_MB:.0f} MB. Para reduzir, diminua LOD ou ARVORES_POR_M2 "
          "na célula de parâmetros e rode esta célula de novo.")

## 7. Visualização interativa

Arraste para girar, use a roda do mouse para aproximar e o botão direito para mover a câmera. **Clique em um prédio ou em uma rua** para ver o nome, o tipo, o número de andares e os estabelecimentos mapeados dentro dele (restaurantes, lanchonetes, lojas), com o link para o OpenStreetMap. Os nomes vêm das tags do OSM, então um prédio sem a tag `name` aparece como sem nome.

In [ ]:
import threading, functools, http.server, socketserver, math
from IPython.display import HTML, IFrame, display

# ---------- nomes e informações de cada elemento do OSM (chave igual à usada no modelo: w<id> ou r<id>) ----------
TIPOS_PT = {"building": "Prédio", "highway": "Via", "leisure": "Lazer", "amenity": "Equipamento",
            "landuse": "Uso do solo", "natural": "Área natural", "barrier": "Barreira", "waterway": "Água"}

def descrever(tags):
    for k, rotulo in TIPOS_PT.items():
        if k in tags:
            return rotulo if tags[k] == "yes" else f"{rotulo} ({tags[k]})"
    return "Elemento do mapa"

DADOS, poligonos = {}, []
for el in list(raiz.iter("way")) + list(raiz.iter("relation")):
    tags = {t.get("k"): t.get("v") for t in el.iter("tag")}
    if not tags:
        continue
    chave = ("w" if el.tag == "way" else "r") + el.get("id")
    DADOS[chave] = {"nome": tags.get("name", ""), "tipo": descrever(tags),
                    "andares": tags.get("building:levels", ""), "uso": tags.get("amenity", ""), "locais": []}
    if el.tag == "way" and "building" in tags:
        pts = [nos[nd.get("ref")] for nd in el.iter("nd") if nd.get("ref") in nos]
        if len(pts) >= 3:
            poligonos.append((chave, pts, min(p[0] for p in pts), max(p[0] for p in pts),
                              min(p[1] for p in pts), max(p[1] for p in pts)))

def dentro(lat, lon, pts):
    c = False
    for (y1, x1), (y2, x2) in zip(pts, pts[1:] + pts[:1]):
        if (y1 > lat) != (y2 > lat) and lon < (x2 - x1) * (lat - y1) / (y2 - y1) + x1:
            c = not c
    return c

# restaurantes, lanchonetes, lojas etc. mapeados como pontos dentro de um prédio
n_locais = 0
for n in raiz.iter("node"):
    tags = {t.get("k"): t.get("v") for t in n.iter("tag")}
    categoria = next((tags[k] for k in ("amenity", "shop", "office", "tourism", "leisure") if k in tags), None)
    if "name" not in tags or categoria is None:
        continue
    lat, lon = float(n.get("lat")), float(n.get("lon"))
    for chave, pts, la0, la1, lo0, lo1 in poligonos:
        if la0 <= lat <= la1 and lo0 <= lon <= lo1 and dentro(lat, lon, pts):
            DADOS[chave]["locais"].append(f"{tags['name']} ({categoria})")
            n_locais += 1
            break
print(f"{sum(1 for d in DADOS.values() if d['nome'])} elementos com nome; "
      f"{n_locais} estabelecimentos associados a prédios.")

# ---------- página do visualizador (three.js) ----------
PAGINA = r"""<!doctype html><html lang="pt-br"><head><meta charset="utf-8">
<title>Campus em 3D</title>
<style>
html,body{margin:0;height:100%;overflow:hidden;font-family:system-ui,-apple-system,sans-serif;background:#cfe3f5}
#info{position:absolute;top:12px;left:12px;max-width:340px;background:rgba(255,255,255,.95);border-radius:8px;
padding:10px 14px;font-size:14px;box-shadow:0 2px 8px rgba(0,0,0,.25)}
#info h3{margin:0 0 4px;font-size:16px}#info p{margin:3px 0;color:#444}#info ul{margin:4px 0 4px 18px;padding:0}
#msg{position:absolute;bottom:10px;left:12px;color:#333;font-size:12px}
</style>
<script type="importmap">{"imports":{
"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js",
"three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}</script>
</head><body>
<div id="info"><h3>__TITULO__</h3><p>Clique em um prédio ou em uma rua para ver o nome.</p></div>
<div id="msg">Carregando o modelo...</div>
<script type="module">
import * as THREE from 'three';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';
import {GLTFLoader} from 'three/addons/loaders/GLTFLoader.js';

const DADOS = __DADOS__;
const info = document.getElementById('info'), msg = document.getElementById('msg');
const renderer = new THREE.WebGLRenderer({antialias:true});
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
renderer.setSize(innerWidth, innerHeight);
document.body.appendChild(renderer.domElement);
const scene = new THREE.Scene();
scene.background = new THREE.Color(0xcfe3f5);
scene.add(new THREE.HemisphereLight(0xffffff, 0x8a8a7a, 1.6));
const sol = new THREE.DirectionalLight(0xffffff, 1.4); sol.position.set(0.5, 1, 0.3); scene.add(sol);
const camera = new THREE.PerspectiveCamera(45, innerWidth / innerHeight, 1, 50000);
const controls = new OrbitControls(camera, renderer.domElement);
controls.maxPolarAngle = Math.PI / 2 - 0.02;
let modelo = null, pedido = true;
controls.addEventListener('change', () => pedido = true);

new GLTFLoader().load('__MODELO__', g => {
  modelo = g.scene; scene.add(modelo);
  const caixa = new THREE.Box3().setFromObject(modelo), c = caixa.getCenter(new THREE.Vector3());
  const t = caixa.getSize(new THREE.Vector3()), d = Math.max(t.x, t.z);
  controls.target.copy(c); camera.position.set(c.x + d * 0.35, c.y + d * 0.35, c.z + d * 0.45);
  controls.update(); msg.textContent = 'Arraste para girar, role para aproximar, botão direito para mover.'; pedido = true;
}, p => { if (p.total) msg.textContent = 'Carregando o modelo... ' + Math.round(100 * p.loaded / p.total) + '%'; },
   e => { msg.textContent = 'Erro ao carregar o modelo: ' + e; });

// ----- seleção por clique -----
const raio = new THREE.Raycaster(), ponto = new THREE.Vector2();
let selecionados = [], baixo = null;
function limpar(){ for (const m of selecionados){ m.material.dispose(); m.material = m.userData._orig; } selecionados = []; }
function destacar(grupo){
  grupo.traverse(m => { if (!m.isMesh || Array.isArray(m.material)) return;
    m.userData._orig = m.material; m.material = m.material.clone();
    if (m.material.emissive){ m.material.emissive.set(0xff6a00); m.material.emissiveIntensity = 0.7; }
    else if (m.material.color) m.material.color.set(0xff6a00);
    selecionados.push(m); });
}
function esc(s){ return String(s).replace(/[&<>"]/g, ch => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[ch])); }
function mostrar(id){
  const d = DADOS[id] || {nome:'', tipo:'Elemento do mapa', andares:'', uso:'', locais:[]};
  const url = 'https://www.openstreetmap.org/' + (id[0] === 'r' ? 'relation/' : 'way/') + id.slice(1);
  let h = '<h3>' + (d.nome ? esc(d.nome) : 'Sem nome no OpenStreetMap') + '</h3><p>' + esc(d.tipo) +
          (d.andares ? ', ' + esc(d.andares) + ' andar(es)' : '') + '</p>';
  if (d.locais && d.locais.length) h += '<p>Neste prédio</p><ul>' + d.locais.map(l => '<li>' + esc(l) + '</li>').join('') + '</ul>';
  h += '<p><a href="' + url + '" target="_blank" rel="noopener">Abrir no OpenStreetMap (' + esc(id) + ')</a></p>';
  info.innerHTML = h;
}
renderer.domElement.addEventListener('pointerdown', e => baixo = [e.clientX, e.clientY]);
renderer.domElement.addEventListener('pointerup', e => {
  if (!modelo || !baixo || Math.hypot(e.clientX - baixo[0], e.clientY - baixo[1]) > 5) return;
  ponto.set(e.clientX / innerWidth * 2 - 1, -(e.clientY / innerHeight) * 2 + 1);
  raio.setFromCamera(ponto, camera);
  limpar(); pedido = true;
  for (const hit of raio.intersectObject(modelo, true)){
    let o = hit.object;
    while (o && !(o.userData && o.userData.osmId)) o = o.parent;
    if (o){ destacar(o); mostrar(o.userData.osmId); return; }
  }
  info.innerHTML = '<h3>__TITULO__</h3><p>Nada identificado neste ponto. Clique em um prédio ou em uma rua.</p>';
});
addEventListener('resize', () => { camera.aspect = innerWidth / innerHeight; camera.updateProjectionMatrix();
  renderer.setSize(innerWidth, innerHeight); pedido = true; });
(function quadro(){ requestAnimationFrame(quadro); if (pedido){ pedido = false; renderer.render(scene, camera); } })();
</script></body></html>"""

html = (PAGINA.replace("__DADOS__", json.dumps(DADOS, ensure_ascii=False).replace("</", "<\\/"))
              .replace("__MODELO__", modelo.name).replace("__TITULO__", "Campus Central da UFRN"))
(modelo.parent / "visualizador.html").write_text(html, encoding="utf-8")

ABRIR = FORCAR_VISUALIZADOR or MODELO_MB <= LIMITE_VISUALIZADOR_MB
if not ABRIR:
    print(f"O modelo tem {MODELO_MB:.0f} MB, acima do limite de {LIMITE_VISUALIZADOR_MB} MB, e o visualizador "
          "não foi aberto para não travar a aba.\nReduza LOD / ARVORES_POR_M2 e converta de novo, ou baixe o pacote "
          "da próxima seção e abra o visualizador.html no seu computador (instruções no fim do notebook).")

PORTA = 8765
class Silencioso(http.server.SimpleHTTPRequestHandler):
    def log_message(self, *a): pass

if ABRIR:
    if "servidor_3d" not in globals():
        handler = functools.partial(Silencioso, directory=str(modelo.parent))
        socketserver.TCPServer.allow_reuse_address = True
        servidor_3d = socketserver.TCPServer(("", PORTA), handler)
        threading.Thread(target=servidor_3d.serve_forever, daemon=True).start()
    try:
        from google.colab import output
        output.serve_kernel_port_as_iframe(PORTA, path="/visualizador.html", height=680)
    except ImportError:
        display(IFrame(f"http://localhost:{PORTA}/visualizador.html", width="100%", height=680))

## 8. Download dos arquivos

In [ ]:
# O pacote principal leva o modelo .glb, o visualizador e os dados do OSM usados
shutil.copy(OSM_FILE, SAIDA_DIR)
pacote = shutil.make_archive(str(BASE / f"{SAIDA_NOME}_3d"), "zip", SAIDA_DIR)
print(f"Pacote: {pacote}  ({os.path.getsize(pacote) / 1e6:.1f} MB)")

pacote_obj = None
if GERAR_OBJ and OBJ_DIR.exists():
    pacote_obj = shutil.make_archive(str(BASE / f"{SAIDA_NOME}_obj"), "zip", OBJ_DIR)
    print(f"Pacote OBJ: {pacote_obj}  ({os.path.getsize(pacote_obj) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(pacote)
    if pacote_obj:
        files.download(pacote_obj)
except ImportError:
    pass

## Próximos passos

* **Melhorar o modelo.** A tabela da seção 5 aponta os prédios sem altura. Completar `building:levels`, `roof:shape` e `building:colour` no editor iD e rodar o notebook de novo já muda bastante o resultado. Os dados novos aparecem na Overpass poucos minutos depois da edição.
* **Abrir no Blender.** Arquivo → Importar → glTF 2.0, apontando para o `.glb`.
* **Publicar na web ou abrir no computador.** O `.glb` e o `visualizador.html` do pacote funcionam juntos em qualquer hospedagem estática, como o GitHub Pages. Para abrir localmente, descompacte o pacote, rode `python -m http.server` dentro da pasta e acesse `http://localhost:8000/visualizador.html` (abrir o arquivo com duplo clique não funciona, porque o navegador bloqueia o carregamento do modelo).
* **Relevo.** O suporte a elevação do OSM2World é experimental e vem desligado, então o campus sai plano.
* **Simuladores.** Com `GERAR_OBJ = True` o notebook também exporta o `.obj`, que serve como cenário com os prédios. As vias onde um veículo vai trafegar precisam ser criadas com as ferramentas do próprio simulador, porque a malha do OSM2World não tem física de pista.